# PrexSyn Baseline

Generates the repeated-PrexSyn-sampling baseline for the benchmarking pipeline.

**Pipeline position:** Stage 2 (Generation) + Stage 4 (Evaluation, baseline arm)

```
ChEMBL NPZ features
      |
      v
Stage 1 -- Build MolSpec for each ChEMBL molecule
      |
      v
Stage 2 -- PrexSyn: generate NUM_SEED_SAMPLES candidates per spec
        -- pick K=1 seed (top Tanimoto to spec) -> baseline_quality, quality_bin
      |
      v
Stage 3 -- Repeated sampling: generate NUM_BASELINE_SAMPLES variants per spec
      |
      v
Stage 4 -- Synthesizability gate (AiZynthFinder)
      |
      v
Stage 5 -- Score variants with scoring_v2 (tanimoto + desirability)
        -- Summarize hit rates by quality_bin x tau_t
        -- Export seeds_for_methods.json for modification notebooks
```

**Kernel:** `prexsyn` conda environment (Python 3.11).

**Output files:**
- `data/baseline_scores.csv` -- per-variant scoring data
- `data/baseline_summary.csv` -- hit rate table by bin and threshold
- `data/seeds_for_methods.json` -- handoff to CReM, mmpdb, LibINVENT, etc.


## 0. Configuration

In [ ]:
import hashlib
import json
import pathlib

# -- paths ---------------------------------------------------------------------
ROOT     = pathlib.Path("..")
DATA_DIR = ROOT / "data"

FEATURES_NPZ           = DATA_DIR / "chembl_features.npz"       # <- input: pre-computed features
SEEDS_JSON             = DATA_DIR / "baseline_seeds.json"        # <- cache: K=1 PrexSyn seeds
VARIANTS_JSON          = DATA_DIR / "baseline_variants.json"     # <- cache: N repeated samples
SCORES_CSV             = DATA_DIR / "baseline_scores.csv"        # <- output: per-variant scores
SUMMARY_CSV            = DATA_DIR / "baseline_summary.csv"       # <- output: bin x threshold table
SEEDS_FOR_METHODS_JSON = DATA_DIR / "seeds_for_methods.json"     # <- output: handoff to modification notebooks

# -- PrexSyn API ---------------------------------------------------------------
PREXSYN_URL = "http://100.65.172.100:8011/sample"  # <- update to your PrexSyn instance

# -- generation parameters -----------------------------------------------------
NUM_SEED_SAMPLES     = 128  # <- candidates per spec for seed selection (K=1 kept)
NUM_BASELINE_SAMPLES = 128  # <- N repeated-sampling variants per spec for the baseline arm

# N_SEEDS controls how many ChEMBL molecules to process.
# Set to None for a full run (all 100); set to an int for a quick smoke-test.
N_SEEDS = None  # <- None = all 100; e.g. 5 for a quick test

# -- evaluation parameters (must match libinvent_modifications.ipynb) ----------
TAU_T_LIST    = [0.6, 0.7, 0.85]  # <- Tanimoto thresholds for hit classification
TAU_D         = 0.8               # <- physicochemical desirability threshold
SYNTH_TIMEOUT = 60                # <- seconds per molecule in AiZynthFinder gate (0 = no limit)
USE_ESPSIM    = False             # <- enable 3D ESPsim supplement (slow, requires conformers)

DATA_DIR.mkdir(parents=True, exist_ok=True)

# -- cache invalidation --------------------------------------------------------
# Hash the generation parameters. If they change, delete cached JSONs so
# PrexSyn re-runs with the new settings instead of returning stale data.
#
# IMPORTANT: The active stock name from config.yml is included in the hash.
# Switching from zinc -> enamine stock automatically invalidates the synth checkpoint.
AIZYNTHFINDER_CONFIG = DATA_DIR / 'aizynthfinder' / 'config.yml'
_active_stock = 'unknown'
if AIZYNTHFINDER_CONFIG.exists():
    import yaml
    _cfg = yaml.safe_load(AIZYNTHFINDER_CONFIG.read_text())
    _active_stock = '|'.join(sorted(_cfg.get('stock', {}).keys()))

_run_params = {
    "N_SEEDS":              N_SEEDS,
    "NUM_SEED_SAMPLES":     NUM_SEED_SAMPLES,
    "NUM_BASELINE_SAMPLES": NUM_BASELINE_SAMPLES,
    "aizynthfinder_stock":  _active_stock,  # <- invalidates checkpoint on stock change
}
_params_hash = hashlib.md5(json.dumps(_run_params, sort_keys=True).encode()).hexdigest()
_meta_file   = DATA_DIR / ".baseline_run_params.json"

_cached_hash = json.loads(_meta_file.read_text()).get("hash") if _meta_file.exists() else None

if _cached_hash != _params_hash:
    deleted = []
    for _f in [SEEDS_JSON, VARIANTS_JSON, SCORES_CSV, SUMMARY_CSV]:
        if _f.exists():
            _f.unlink()
            deleted.append(_f.name)
    if deleted:
        print(f"[cache] Parameters changed -- deleted: {', '.join(deleted)}")
    _meta_file.write_text(json.dumps({"hash": _params_hash, "params": _run_params}, indent=2))

print(f"N_SEEDS              : {N_SEEDS if N_SEEDS is not None else 'all (100)'}")
print(f"NUM_SEED_SAMPLES     : {NUM_SEED_SAMPLES}")
print(f"NUM_BASELINE_SAMPLES : {NUM_BASELINE_SAMPLES}")
print(f"TAU_T_LIST           : {TAU_T_LIST}")
print(f"TAU_D                : {TAU_D}")
print(f"AiZynthFinder stock  : {_active_stock}")
print("Config OK")


## 1. Imports

In [ ]:
import json
import sys
import time
import urllib.error
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Add repo root to path so src/ modules are importable
sys.path.insert(0, str(ROOT))

from src.evaluation.scoring_v2 import (
    BIN_LABELS,
    DEFAULT_TAU_D,
    DEFAULT_TAU_T_LIST,
    make_spec,
    score_batch,
    summarize,
    classify_hits,
    tanimoto_to_spec,
)
from rdkit import Chem

warnings.filterwarnings("ignore")
print("Imports OK")

## 2. Load ChEMBL features

The NPZ file is produced by `featurize_chembl.py` and contains pre-computed
ECFP4, FCFP4, RDKit descriptor, and BRICS fingerprint arrays.
Each row corresponds to one ChEMBL molecule.

In [ ]:
assert FEATURES_NPZ.exists(), (
    f"Features file not found: {FEATURES_NPZ}\n"
    "Run: python src/generation/main.py featurize"
)

data = np.load(FEATURES_NPZ, allow_pickle=True)

smiles_arr       = data["smiles"]
ecfp4_arr        = data["ecfp4"]
fcfp4_arr        = data["fcfp4"]
rdkit_vals_arr   = data["rdkit_desc_values"]
rdkit_names      = data["rdkit_desc_names"].tolist()
brics_fps_arr    = data["brics_fps"]
brics_exists_arr = data["brics_exists"]

N_TOTAL = len(smiles_arr) if N_SEEDS is None else min(N_SEEDS, len(smiles_arr))

print(f"Loaded {len(smiles_arr)} molecules from {FEATURES_NPZ}")
print(f"Processing {N_TOTAL} / {len(smiles_arr)} molecules (N_SEEDS={N_SEEDS})")
print(f"Feature arrays: ecfp4={ecfp4_arr.shape}, rdkit_vals={rdkit_vals_arr.shape}")

## 3. Build MolSpecs

Each ChEMBL SMILES is converted to a `MolSpec` containing the ECFP4 fingerprint
and target descriptor values (MW, CLogP, TPSA, HBD, HBA, RotBonds).
The ChEMBL molecule itself is not an optimization target.

In [ ]:
specs     = []   # MolSpec objects
spec_smi  = []   # corresponding SMILES strings
skip_idx  = []   # indices where make_spec failed

for i in range(N_TOTAL):
    smi  = str(smiles_arr[i])
    spec = make_spec(smi, generate_conformer=USE_ESPSIM)
    if spec is None:
        skip_idx.append(i)
        continue
    specs.append(spec)
    spec_smi.append(smi)

print(f"Built {len(specs)} MolSpecs  ({len(skip_idx)} skipped: invalid SMILES)")

# Preview descriptor distribution
desc_df = pd.DataFrame([
    {"mw": s.mw, "clogp": s.clogp, "tpsa": s.tpsa,
     "hbd": s.hbd, "hba": s.hba, "rot_bonds": s.rot_bonds}
    for s in specs
])
desc_df.describe().round(2)

## 4. PrexSyn generation -- seed selection (K = 1)

For each spec, call the PrexSyn `/sample` endpoint with `NUM_SEED_SAMPLES=256`
candidates. The top candidate by ECFP4 Tanimoto to the spec is the **seed**
that will be passed to all modification methods.

The seed's Tanimoto (`baseline_quality`) determines the difficulty bin.

In [ ]:
def _call_prexsyn(payload: dict, url: str = PREXSYN_URL, retries: int = 3) -> dict | None:
    """POST to PrexSyn /sample; return parsed JSON or None on failure."""
    for attempt in range(retries):
        try:
            req  = urllib.request.Request(
                url,
                data=json.dumps(payload).encode(),
                headers={"Content-Type": "application/json"},
            )
            resp = urllib.request.urlopen(req, timeout=120)
            return json.loads(resp.read())
        except urllib.error.HTTPError as e:
            print(f"  HTTP {e.code}: {e.read().decode()[:120]}")
            return None
        except Exception as exc:
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
            else:
                print(f"  Failed after {retries} attempts: {exc}")
                return None
    return None


def _build_payload(i: int, num_samples: int) -> dict:
    """Assemble the JSON payload for molecule index i."""
    return {
        "ecfp4":             ecfp4_arr[i].tolist(),
        "fcfp4":             fcfp4_arr[i].tolist(),
        "rdkit_desc_values": rdkit_vals_arr[i].tolist(),
        "rdkit_desc_names":  rdkit_names,
        "brics_fps":         brics_fps_arr[i].tolist(),
        "brics_exists":      brics_exists_arr[i].tolist(),
        "source_smiles":     str(smiles_arr[i]),
        "num_samples":       num_samples,
    }


print("Payload builder ready.")

In [ ]:
if SEEDS_JSON.exists():
    seeds = json.loads(SEEDS_JSON.read_text(encoding="utf-8"))
    print(f"Loaded {len(seeds)} cached seeds from {SEEDS_JSON} (skipping API calls)")
else:
    seeds = []
    valid_indices = [i for i in range(N_TOTAL) if i not in skip_idx]

    for spec_idx, arr_idx in enumerate(tqdm(valid_indices, desc="PrexSyn seed generation")):
        spec = specs[spec_idx]

        payload  = _build_payload(arr_idx, num_samples=NUM_SEED_SAMPLES)
        response = _call_prexsyn(payload)

        if response is None or not response.get("generated_smiles"):
            seeds.append({
                "spec_smiles":      spec_smi[spec_idx],
                "seed_smiles":      None,
                "baseline_quality": 0.0,
                "quality_bin":      "<0.5",
                "num_candidates":   0,
                "raw_candidates":   [],
            })
            continue

        candidates = response["generated_smiles"]

        # Pick K=1: highest Tanimoto to spec fingerprint
        best_smi, best_tan = None, -1.0
        for smi in candidates:
            if "." in smi:
                continue
            mol = Chem.MolFromSmiles(smi)
            if mol is None:
                continue
            tan = tanimoto_to_spec(mol, spec)
            if tan > best_tan:
                best_tan, best_smi = tan, smi

        from src.evaluation.scoring_v2 import _assign_bin
        qbin = _assign_bin(best_tan) if best_smi else "<0.5"

        seeds.append({
            "spec_smiles":      spec_smi[spec_idx],
            "seed_smiles":      best_smi,
            "baseline_quality": round(best_tan, 4),
            "quality_bin":      qbin,
            "num_candidates":   len(candidates),
            "raw_candidates":   candidates,
        })

    SEEDS_JSON.write_text(json.dumps(seeds, indent=2), encoding="utf-8")
    print(f"Saved {len(seeds)} seed records -> {SEEDS_JSON}")

In [ ]:
# -- Baseline quality distribution --
seeds_df = pd.DataFrame(seeds)
seeds_df = seeds_df[seeds_df["seed_smiles"].notna()]

print(f"Valid seeds: {len(seeds_df)} / {len(seeds)}")
print("\nBaseline quality distribution:")
print(seeds_df["baseline_quality"].describe().round(3))
print("\nBin counts:")
print(seeds_df["quality_bin"].value_counts().sort_index())

fig, ax = plt.subplots(figsize=(7, 3))
seeds_df["baseline_quality"].hist(bins=20, ax=ax, color="steelblue", edgecolor="white")
for threshold in [0.5, 0.7, 0.85]:
    ax.axvline(threshold, color="tomato", linewidth=1.2, linestyle="--")
ax.set_xlabel("Baseline quality (Tanimoto, seed vs spec)")
ax.set_ylabel("Count")
ax.set_title("PrexSyn baseline quality distribution")
plt.tight_layout()
plt.show()

## 5. Repeated-sampling baseline generation (N draws)

For each spec, generate `NUM_BASELINE_SAMPLES` additional PrexSyn candidates
from the same property specification. These form the repeated-sampling baseline
that all modification methods are compared against.

The same spec payload is reused; PrexSyn's sampling is stochastic so the
returned set differs from the seed-generation batch.

In [ ]:
arr_idx_map = {spec_idx: arr_idx for spec_idx, arr_idx in enumerate(
    [i for i in range(N_TOTAL) if i not in skip_idx]
)}

if VARIANTS_JSON.exists():
    baseline_variants = json.loads(VARIANTS_JSON.read_text(encoding="utf-8"))
    print(f"Loaded {len(baseline_variants)} cached variant batches from {VARIANTS_JSON} (skipping API calls)")
else:
    baseline_variants = []

    for spec_idx, row in enumerate(tqdm(seeds, desc="Repeated-sampling baseline")):
        if row["seed_smiles"] is None:
            continue

        arr_idx  = arr_idx_map[spec_idx]
        payload  = _build_payload(arr_idx, num_samples=NUM_BASELINE_SAMPLES)
        response = _call_prexsyn(payload)

        variants = response.get("generated_smiles", []) if response else []

        # Deduplicate by canonical SMILES; exclude seed
        seen    = {row["seed_smiles"]}
        deduped = []
        for smi in variants:
            mol   = Chem.MolFromSmiles(smi)
            canon = Chem.MolToSmiles(mol) if mol else None
            if canon and canon not in seen:
                seen.add(canon)
                deduped.append(canon)

        baseline_variants.append({
            "spec_smiles":      row["spec_smiles"],
            "baseline_quality": row["baseline_quality"],
            "quality_bin":      row["quality_bin"],
            "seed_smiles":      row["seed_smiles"],
            "variants":         deduped,
            "n_raw":            len(variants),
            "n_deduped":        len(deduped),
        })

    VARIANTS_JSON.write_text(json.dumps(baseline_variants, indent=2), encoding="utf-8")
    total_variants = sum(e["n_deduped"] for e in baseline_variants)
    print(f"Saved {len(baseline_variants)} specs, {total_variants} total variants -> {VARIANTS_JSON}")

In [ ]:
# -- Variant yield summary --
yield_df = pd.DataFrame([
    {"quality_bin": e["quality_bin"],
     "n_raw":       e["n_raw"],
     "n_deduped":   e["n_deduped"]}
    for e in baseline_variants
])

print("Variant yield by difficulty bin:")
print(
    yield_df.groupby("quality_bin")[["n_raw", "n_deduped"]]
    .agg(["mean", "min", "max"])
    .round(1)
)

## 6. Synthesizability gate (AiZynthFinder)

Each baseline variant must pass retrosynthetic analysis before property
conservation scoring. The `synthesizability.py` module wraps AiZynthFinder.

> **Note:** AiZynthFinder runs in the `prexsyn` environment. This cell
> calls `src/evaluation/synthesizability.py` directly if the model files
> are present; otherwise it skips the gate and records all variants as
> unverified (for dry-run / development purposes).

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

AIZYNTHFINDER_CONFIG = DATA_DIR / "aizynthfinder" / "config.yml"

USE_AIZYNTHFINDER = AIZYNTHFINDER_CONFIG.exists()
SYNTH_CHECKPOINT  = DATA_DIR / "baseline_synth_checkpoint.json"

if USE_AIZYNTHFINDER:
    # Import per-worker initializer and scoring function for ProcessPoolExecutor.
    # worker_init() loads AiZynthFinder once per worker process (not once per molecule),
    # avoiding the dominant model-reload overhead in sequential execution.
    from src.evaluation.synth_parallel import worker_init, score_one
    N_WORKERS = 8
    print(f"AiZynthFinder config found: {AIZYNTHFINDER_CONFIG}")
    print(f"Parallel workers: {N_WORKERS} (logical CPUs: {os.cpu_count()})")
else:
    print(
        f"[WARN] AiZynthFinder config not found at {AIZYNTHFINDER_CONFIG}.\n"
        "       Skipping synthesizability gate -- all variants treated as synthesizable.\n"
        "       Set AIZYNTHFINDER_CONFIG to enable Gate 1."
    )

In [ ]:
from concurrent.futures import TimeoutError as FutureTimeoutError

synth_results: dict[str, bool] = {}
total_variants_count = sum(len(e["variants"]) for e in baseline_variants)

# Flatten to a deduplicated list of SMILES across all specs
all_variant_smiles = list(dict.fromkeys(
    smi for entry in baseline_variants for smi in entry["variants"]
))

if USE_AIZYNTHFINDER:
    # Resume from checkpoint if available
    if SYNTH_CHECKPOINT.exists():
        with open(SYNTH_CHECKPOINT) as _f:
            synth_results = json.load(_f)
        print(f"Resumed checkpoint: {len(synth_results)} molecules already scored")

    remaining = [s for s in all_variant_smiles if s not in synth_results]
    print(f"Total unique variants : {len(all_variant_smiles)}")
    print(f"Already scored        : {len(all_variant_smiles) - len(remaining)}")
    print(f"Remaining             : {len(remaining)}")

    _timeout    = SYNTH_TIMEOUT if SYNTH_TIMEOUT > 0 else None
    _save_every = 50
    n_timeout   = 0

    if remaining:
        with ProcessPoolExecutor(
            max_workers=N_WORKERS,
            initializer=worker_init,
            initargs=(str(AIZYNTHFINDER_CONFIG),),
        ) as executor:
            futures = {executor.submit(score_one, smi): smi for smi in remaining}

            with tqdm(total=len(remaining), desc="Synthesizability gate", unit="mol") as pbar:
                for future in as_completed(futures):
                    smi = futures[future]
                    try:
                        _, is_solved = future.result(timeout=_timeout)
                    except FutureTimeoutError:
                        is_solved = False
                        n_timeout += 1
                    except Exception:
                        is_solved = False
                    synth_results[smi] = is_solved
                    pbar.update(1)

                    if len(synth_results) % _save_every == 0:
                        with open(SYNTH_CHECKPOINT, "w") as _f:
                            json.dump(synth_results, _f)

        # Final checkpoint save
        with open(SYNTH_CHECKPOINT, "w") as _f:
            json.dump(synth_results, _f)
        if n_timeout:
            print(f"[WARN] {n_timeout} molecules timed out (>{SYNTH_TIMEOUT}s) -- marked not synthesizable")

else:
    # No AiZynthFinder config -- treat all as synthesizable (dev/dry-run)
    synth_results = {smi: True for smi in all_variant_smiles}

# Reconstruct per-spec structure from flat results
synth_filtered = []
for entry in baseline_variants:
    passed = [s for s in entry["variants"] if synth_results.get(s, False)]
    synth_filtered.append({**entry, "variants": passed, "n_synth": len(passed)})

total_synth = sum(e["n_synth"] for e in synth_filtered)
total_raw   = sum(len(e["variants"]) for e in baseline_variants)
rate = f"{100 * total_synth / total_raw:.1f}%" if (total_raw and USE_AIZYNTHFINDER) else "N/A (gate skipped)"
print(f"After synthesizability gate: {total_synth} / {total_raw} variants pass ({rate})")

## 7. Score with scoring_v2

Each synthesizable baseline variant is scored against its property specification:
- **Substructural conservation:** ECFP4 Tanimoto vs. spec fingerprint  
- **Physicochemical conservation:** spec-relative desirability (geometric mean of 6 descriptor deltas)

In [ ]:
from rdkit import DataStructs
from rdkit.Chem import rdMolDescriptors

# Build lookup tables for fast per-entry access
spec_lookup    = {smi: spec for smi, spec in zip(spec_smi, specs)}
seed_fp_lookup = {}
for entry in synth_filtered:
    seed_smi = entry.get("seed_smiles")
    if seed_smi:
        mol = Chem.MolFromSmiles(seed_smi)
        if mol:
            seed_fp_lookup[entry["spec_smiles"]] = (
                rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
            )

all_score_dfs = []

for entry in tqdm(synth_filtered, desc="Scoring baseline variants"):
    spec = spec_lookup.get(entry["spec_smiles"])
    if spec is None or not entry["variants"]:
        continue

    batch_df = score_batch(
        variants         = entry["variants"],
        spec             = spec,
        baseline_quality = entry["baseline_quality"],
        method           = "baseline",
        compute_espsim   = USE_ESPSIM,
    )

    if not batch_df.empty:
        # bioactive_proximity: ECFP4 Tanimoto to the PrexSyn seed (diagnostic)
        seed_fp = seed_fp_lookup.get(entry["spec_smiles"])
        if seed_fp is not None:
            def _prox(smi, _fp=seed_fp):
                mol = Chem.MolFromSmiles(smi)
                if mol is None:
                    return None
                vfp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
                return round(float(DataStructs.TanimotoSimilarity(vfp, _fp)), 4)
            batch_df["bioactive_proximity"] = batch_df["variant_smiles"].map(_prox)
        else:
            batch_df["bioactive_proximity"] = None

        all_score_dfs.append(batch_df)

scores_df = pd.concat(all_score_dfs, ignore_index=True) if all_score_dfs else pd.DataFrame()

scores_df.to_csv(SCORES_CSV, index=False)
print(f"Scored {len(scores_df)} variants -> {SCORES_CSV}")
scores_df.head()

## 8. Summary statistics

Bin-stratified hit rates at all three Tanimoto thresholds.  
These values define the baseline against which all modification methods are measured.

In [ ]:
if scores_df.empty:
    print("No scored variants -- check upstream steps.")
else:
    summary_df = summarize(scores_df, tau_t_list=TAU_T_LIST, tau_d=TAU_D)
    summary_df.to_csv(SUMMARY_CSV, index=False)
    print(f"Summary ({len(summary_df)} rows) -> {SUMMARY_CSV}")
    display(summary_df)

In [ ]:
# -- Hit rate by bin x threshold --
if not scores_df.empty:
    fig, axes = plt.subplots(1, len(TAU_T_LIST), figsize=(5 * len(TAU_T_LIST), 4), sharey=True)

    for ax, tau_t in zip(axes, TAU_T_LIST):
        sub = summary_df[summary_df["tau_t"] == tau_t]
        ax.bar(sub["quality_bin"], sub["hit_rate"], color="steelblue", edgecolor="white")
        ax.set_title(f"tau_t={tau_t}")
        ax.set_xlabel("Baseline quality bin")
        ax.set_ylabel("Hit rate")
        ax.set_ylim(0, 1)
        for bar, val in zip(ax.patches, sub["hit_rate"]):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.02,
                f"{val:.2f}", ha="center", va="bottom", fontsize=9
            )

    fig.suptitle("Baseline hit rate by difficulty bin", fontsize=13)
    plt.tight_layout()
    plt.savefig(DATA_DIR / "baseline_hit_rates.png", dpi=150)
    plt.show()

In [ ]:
# -- Score distributions --
if not scores_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

    for qbin, grp in scores_df.groupby("quality_bin"):
        grp["tanimoto"].plot.kde(ax=axes[0], label=qbin)
        grp["desirability"].plot.kde(ax=axes[1], label=qbin)

    axes[0].set_title("Tanimoto distribution by bin")
    axes[0].set_xlabel("Tanimoto (variant vs spec)")
    for tau_t in TAU_T_LIST:
        axes[0].axvline(tau_t, linewidth=0.8, linestyle="--", color="gray")

    axes[1].set_title("Desirability distribution by bin")
    axes[1].set_xlabel("Spec-relative desirability")
    axes[1].axvline(TAU_D, linewidth=0.8, linestyle="--", color="gray", label=f"tau_d={TAU_D}")

    for ax in axes:
        ax.legend(title="Bin", fontsize=8)
        ax.set_ylabel("Density")

    plt.tight_layout()
    plt.savefig(DATA_DIR / "baseline_score_distributions.png", dpi=150)
    plt.show()

## 9. Saturation analysis

Check whether additional PrexSyn samples continue to find new unique hits, or
whether the baseline saturates. This informs the choice of N for all methods.

For each spec, compute cumulative unique hits as a function of the number of
variants considered (in order of generation).

In [ ]:
if not scores_df.empty:
    TAU_T_PILOT = 0.6  # use most permissive threshold for saturation check

    # For each spec, track cumulative unique hits as N increases
    saturation_curves = []

    for spec_smiles, grp in scores_df.groupby("spec_smiles"):
        grp = grp.reset_index(drop=True)
        grp["is_hit"] = classify_hits(grp, tau_t=TAU_T_PILOT, tau_d=TAU_D)

        seen_hits  = set()
        curve      = []
        for _, row in grp.iterrows():
            if row["is_hit"]:
                seen_hits.add(row["variant_smiles"])
            curve.append(len(seen_hits))

        saturation_curves.append(curve)

    # Pad curves to the same length and average
    max_len    = max(len(c) for c in saturation_curves)
    padded     = [c + [c[-1]] * (max_len - len(c)) for c in saturation_curves]
    mean_curve = np.mean(padded, axis=0)

    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(range(1, max_len + 1), mean_curve, color="steelblue")
    ax.set_xlabel("Number of variants sampled")
    ax.set_ylabel("Mean cumulative unique hits")
    ax.set_title(f"Baseline saturation curve (tau_t={TAU_T_PILOT}, tau_d={TAU_D})")
    plt.tight_layout()
    plt.savefig(DATA_DIR / "baseline_saturation.png", dpi=150)
    plt.show()

    # Marginal gain in last 10% of samples
    cutoff      = int(0.9 * max_len)
    marginal_10 = mean_curve[-1] - mean_curve[cutoff]
    print(f"Mean unique hits at N={max_len}: {mean_curve[-1]:.2f}")
    print(f"Marginal gain in last 10% of samples: {marginal_10:.2f} hits")

## 10. Export seed file for modification methods

Export the seed SMILES for each spec in the format expected by the
modification method notebooks (CReM, mmpdb, JT-VAE, LibINVENT).

In [ ]:
# Format for modification method notebooks:
# [
#   {
#     "spec_smiles":      "...",
#     "seed_smiles":      "...",
#     "baseline_quality": 0.72,
#     "quality_bin":      "0.7-0.85",
#     "methods": {"baseline": ["...", ...]}
#   },
#   ...
# ]

seeds_for_methods = []

for entry in synth_filtered:
    seed_row = next(
        (s for s in seeds if s["spec_smiles"] == entry["spec_smiles"]), None
    )
    if seed_row is None or seed_row["seed_smiles"] is None:
        continue

    seeds_for_methods.append({
        "spec_smiles":      entry["spec_smiles"],
        "seed_smiles":      seed_row["seed_smiles"],
        "baseline_quality": entry["baseline_quality"],
        "quality_bin":      entry["quality_bin"],
        "methods": {
            "baseline": entry["variants"]   # synthesizability-filtered
        },
    })

SEEDS_FOR_METHODS_JSON = DATA_DIR / "seeds_for_methods.json"
SEEDS_FOR_METHODS_JSON.write_text(json.dumps(seeds_for_methods, indent=2))

print(f"Exported {len(seeds_for_methods)} specs with seed + baseline variants")
print(f"-> {SEEDS_FOR_METHODS_JSON}")
print("\nThis file is the input for all modification method notebooks.")

## 11. Final summary

Print a concise overview of what was produced.

In [ ]:
print("=" * 60)
print("PrexSyn Baseline -- Final Summary")
print("=" * 60)
print(f"  Specs processed:          {len(seeds_for_methods)}")
print(f"  Total baseline variants:  {total_raw}")
print(f"  After synth gate:         {total_synth}")
print(f"  Scored variants:          {len(scores_df)}")
print()
print("Outputs:")
for path in [SEEDS_JSON, VARIANTS_JSON, SCORES_CSV, SUMMARY_CSV, SEEDS_FOR_METHODS_JSON]:
    size = path.stat().st_size if path.exists() else 0
    print(f"  {path.name:<35s}  {size:>8,} bytes")
print()

if not scores_df.empty:
    print("Hit rates at tau_d = 0.8:")
    pivot = summary_df.pivot_table(
        index="quality_bin", columns="tau_t", values="hit_rate"
    ).round(3)
    print(pivot.to_string())